# Classic Black-Box Testing Exercise

In this exercise you practice **black-box** test design: derive tests from **requirements** (inputs/outputs, rules) **without** relying on knowledge of how the code is implemented internally.

The system under test takes **three integer parameters** (sensor readings of a small reactor safety monitor), which lets us practice combinatorial techniques on **more than two factors**.

You will work through:
- **Equivalence partitioning (EP)** — group each input into classes and pick representatives.
- **Boundary value analysis (BVA)** — stress values at/near partition edges, including the edge of the dangerous two-sensor interaction.
- **Combinatorial interaction testing (CIT)** — build the **full Cartesian product** of EP levels for all three factors, then **reduce** it to a minimal **pairwise (2-way)** set and **verify** the coverage.

Fill in the **`TODO`** sections and run the cells **from top to bottom** (helper functions are defined in earlier code cells and reused below).

## Software under Test (SuT) — specification

Function name: **`reactor_alarm(temp, pressure, coolant_loss)`** → returns a `str`.

A simplified **reactor safety monitor**. Each argument is an **integer sensor reading**.

**Inputs & valid ranges** — any reading outside its range makes the whole result **`"INVALID"`**:
- `temp` — core temperature in °C. Valid **0–1000** inclusive.
- `pressure` — vessel pressure in kPa. Valid **0–300** inclusive.
- `coolant_loss` — coolant lost in %. Valid **0–100** inclusive.

**Per-sensor severity** (`0` = normal, `1` = warn, `2` = critical):

| sensor | normal (0) | warn (1) | critical (2) |
|---|---|---|---|
| `temp` | 0–600 | 601–800 | 801–1000 |
| `pressure` | 0–180 | 181–240 | 241–300 |
| `coolant_loss` | 0–20 | 21–50 | 51–100 |

**Alarm output** — the alarm level is the **maximum** severity across the three sensors:
- `0` → **`"OK"`**, `1` → **`"WARN"`**, `2` → **`"CRITICAL"`**.

**Dangerous interaction (override)**
- If **`temp` is critical AND `coolant_loss` is critical** (both severity `2`), the result must be **`"SHUTDOWN"`** instead of `"CRITICAL"` — a hot core that is also losing coolant must trip a full shutdown. (Note: this is a **2-way** interaction — exactly the kind pairwise testing targets.)

Design tests from this specification. The next cell contains an implementation only so you can **execute** tests; treat the box as black when choosing cases.

In [13]:
def reactor_alarm(temp: int, pressure: int, coolant_loss: int) -> str:
    # Validity: any reading outside its sensor range is INVALID.
    if not (0 <= temp <= 1000):
        return "INVALID"
    if not (0 <= pressure <= 300):
        return "INVALID"
    if not (0 <= coolant_loss <= 100):
        return "INVALID"

    # Per-sensor severity: 0 = normal, 1 = warn, 2 = critical.
    def temp_sev(t: int) -> int:
        return 0 if t <= 600 else (1 if t <= 800 else 2)

    def pressure_sev(p: int) -> int:
        return 0 if p <= 180 else (1 if p <= 240 else 2)

    def coolant_sev(c: int) -> int:
        return 0 if c <= 20 else (1 if c <= 50 else 2)

    ts, ps, cs = temp_sev(temp), pressure_sev(pressure), coolant_sev(coolant_loss)

    # Dangerous 2-way interaction: a critically hot core that is also losing
    # coolant critically must trip a full SHUTDOWN, whatever the pressure is.
    if ts == 2 and cs == 2:
        return "SHUTDOWN"

    level = max(ts, ps, cs)
    return {0: "OK", 1: "WARN", 2: "CRITICAL"}[level]


## Equivalence partitioning (EP)

EP refines **one input at a time**: split that input's domain into **equivalence classes** (values treated the same) and pick **one representative per class**. To isolate a single parameter's effect, **hold the other two fixed at a normal baseline** (`temp = 300`, `pressure = 100`, `coolant_loss = 10`) and vary only the parameter under study.

Each parameter has **5 classes**: *invalid-low / normal / warn / critical / invalid-high*.

> The `SHUTDOWN` rule needs **two** sensors critical at once — that is an **interaction**, not a single-parameter class, so with the other two held normal it never appears here. We cover it in **CIT** below.

**TODO**
- For `temp`, then `pressure`, then `coolant_loss`: hold the other two at the baseline and write **one test per class**.
- Write cases as `(temp, pressure, coolant_loss, expected)` tuples.
- Run `run_tests` below.

In [14]:
def run_tests(cases: list[tuple[int, int, int, str]]) -> None:
    for temp, pressure, coolant_loss, expected in cases:
        got = reactor_alarm(temp, pressure, coolant_loss)
        ok = got == expected
        print(f"temp={temp:>4}, pressure={pressure:>3}, coolant_loss={coolant_loss:>3} "
              f"-> {got:<8} (expected {expected:<8}) {'OK' if ok else 'FAIL'}")


# EP refines ONE parameter at a time; the other two stay at a normal baseline
# (temp=300, pressure=100, coolant_loss=10). Each parameter has 5 classes:
# invalid-low / normal / warn / critical / invalid-high.
ep_cases: list[tuple[int, int, int, str]] = [
    # (temp, pressure, coolant_loss, expected)
    # --- partition `temp`  (pressure & coolant_loss fixed normal) ---
    (-1,   100,  10, "INVALID"),    # invalid: below 0
    (300,  100,  10, "OK"),         # normal
    (700,  100,  10, "WARN"),       # warn
    (900,  100,  10, "CRITICAL"),   # critical
    (1001, 100,  10, "INVALID"),    # invalid: above 1000
    # --- partition `pressure`  (temp & coolant_loss fixed normal) ---
    (300,   -1,  10, "INVALID"),    # invalid: below 0
    (300,  100,  10, "OK"),         # normal
    (300,  210,  10, "WARN"),       # warn
    (300,  270,  10, "CRITICAL"),   # critical
    (300,  301,  10, "INVALID"),    # invalid: above 300
    # --- partition `coolant_loss`  (temp & pressure fixed normal) ---
    (300,  100,  -1, "INVALID"),    # invalid: below 0
    (300,  100,  10, "OK"),         # normal
    (300,  100,  35, "WARN"),       # warn
    (300,  100,  75, "CRITICAL"),   # critical
    (300,  100, 101, "INVALID"),    # invalid: above 100
]

run_tests(ep_cases)


temp=  -1, pressure=100, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp= 300, pressure=100, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 700, pressure=100, coolant_loss= 10 -> WARN     (expected WARN    ) OK
temp= 900, pressure=100, coolant_loss= 10 -> CRITICAL (expected CRITICAL) OK
temp=1001, pressure=100, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp= 300, pressure= -1, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp= 300, pressure=100, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 300, pressure=210, coolant_loss= 10 -> WARN     (expected WARN    ) OK
temp= 300, pressure=270, coolant_loss= 10 -> CRITICAL (expected CRITICAL) OK
temp= 300, pressure=301, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp= 300, pressure=100, coolant_loss= -1 -> INVALID  (expected INVALID ) OK
temp= 300, pressure=100, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 300, pressure=100, coolant_loss= 35 -> WARN     (expected WARN    ) OK

## Boundary value analysis (BVA)

Like EP, BVA refines **one parameter at a time** — the other two stay at the normal baseline (`temp = 300`, `pressure = 100`, `coolant_loss = 10`). Place tests at the **edges** of that parameter's partitions: just inside/outside the valid range and on **both sides** of every severity threshold.

Thresholds to probe (value just below / at / just above):
- `temp`: **0**, **600/601**, **800/801**, **1000/1001**
- `pressure`: **0**, **180/181**, **240/241**, **300/301**
- `coolant_loss`: **0**, **20/21**, **50/51**, **100/101**

> The two-sensor `SHUTDOWN` edge (`temp` *and* `coolant_loss` crossing into critical together) is **not** a single-parameter boundary — it is an interaction, covered in **CIT**.

**TODO**
- For each sensor in turn, hold the other two at the baseline and add boundary cases around every threshold.

In [15]:
# BVA refines ONE parameter at a time (other two held at the normal baseline
# temp=300, pressure=100, coolant_loss=10), probing the edge of each partition.
bva_cases: list[tuple[int, int, int, str]] = [
    # --- temp boundaries  (pressure & coolant_loss held normal) ---
    (-1,   100,  10, "INVALID"),
    (0,    100,  10, "OK"),
    (600,  100,  10, "OK"),
    (601,  100,  10, "WARN"),
    (800,  100,  10, "WARN"),
    (801,  100,  10, "CRITICAL"),
    (1000, 100,  10, "CRITICAL"),
    (1001, 100,  10, "INVALID"),
    # --- pressure boundaries  (temp & coolant_loss held normal) ---
    (300,   -1,  10, "INVALID"),
    (300,    0,  10, "OK"),
    (300,  180,  10, "OK"),
    (300,  181,  10, "WARN"),
    (300,  240,  10, "WARN"),
    (300,  241,  10, "CRITICAL"),
    (300,  300,  10, "CRITICAL"),
    (300,  301,  10, "INVALID"),
    # --- coolant_loss boundaries  (temp & pressure held normal) ---
    (300,  100,  -1, "INVALID"),
    (300,  100,   0, "OK"),
    (300,  100,  20, "OK"),
    (300,  100,  21, "WARN"),
    (300,  100,  50, "WARN"),
    (300,  100,  51, "CRITICAL"),
    (300,  100, 100, "CRITICAL"),
    (300,  100, 101, "INVALID"),
]

run_tests(bva_cases)


temp=  -1, pressure=100, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp=   0, pressure=100, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 600, pressure=100, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 601, pressure=100, coolant_loss= 10 -> WARN     (expected WARN    ) OK
temp= 800, pressure=100, coolant_loss= 10 -> WARN     (expected WARN    ) OK
temp= 801, pressure=100, coolant_loss= 10 -> CRITICAL (expected CRITICAL) OK
temp=1000, pressure=100, coolant_loss= 10 -> CRITICAL (expected CRITICAL) OK
temp=1001, pressure=100, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp= 300, pressure= -1, coolant_loss= 10 -> INVALID  (expected INVALID ) OK
temp= 300, pressure=  0, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 300, pressure=180, coolant_loss= 10 -> OK       (expected OK      ) OK
temp= 300, pressure=181, coolant_loss= 10 -> WARN     (expected WARN    ) OK
temp= 300, pressure=240, coolant_loss= 10 -> WARN     (expected WARN    ) OK

## Combinatorial interaction testing (CIT)

EP and BVA each varied **one** parameter while holding the other two fixed, so they never exercised how the three sensors **interact** (e.g. the `SHUTDOWN` rule, which needs `temp` *and* `coolant_loss` critical together). CIT fills that gap by **combining** factor levels. It has three steps:

1. **Full Cartesian product** — take the **EP representative levels** of each factor (3 × 3 × 3) and run **every** combination. That is the exhaustive grid: **27** tests.
2. **Pairwise (2-way) reduction** — most interaction faults are triggered by **a pair** of factor values, not by all three at once (our `SHUTDOWN` rule is exactly a 2-way interaction). A **pairwise covering array** keeps every 2-way combination while using far fewer tests.
3. **Verify** — check that the reduced set still covers **all** 2-way combinations.

### Step 1 — full Cartesian product (EP levels)

Pick **one interior representative per band** for each factor (avoid threshold values — those were covered by BVA):

| factor | normal | warn | critical |
|---|---|---|---|
| `temp` | 300 | 700 | 900 |
| `pressure` | 100 | 210 | 270 |
| `coolant_loss` | 10 | 35 | 75 |

Combine **all** of them with `itertools.product` → **3 × 3 × 3 = 27** tests. This is the exhaustive baseline we will shrink in Step 2.

In [16]:
import itertools

# Score axis from EP: one interior representative per band, for each of the 3 factors.
factors: dict[str, list[int]] = {
    "temp":         [300, 700, 900],   # normal / warn / critical interiors
    "pressure":     [100, 210, 270],
    "coolant_loss": [10,  35,  75],
}


def run_grid(combos: list[tuple[int, int, int]], heading: str) -> None:
    """Run reactor_alarm on every combination in `combos` and print the results."""
    print("=" * 64)
    print(heading)
    print("=" * 64)
    print("number of tests:", len(combos))
    print("--- run ---")
    for temp, pressure, coolant_loss in combos:
        result = reactor_alarm(temp, pressure, coolant_loss)
        print(f"(temp={temp:>3}, pressure={pressure:>3}, coolant_loss={coolant_loss:>2}) -> {result}")
    print()


# Full Cartesian product: every level of every factor combined with every other.
full_grid = list(itertools.product(factors["temp"], factors["pressure"], factors["coolant_loss"]))
run_grid(full_grid, "CIT — full Cartesian product (3 x 3 x 3 = 27 tests)")


CIT — full Cartesian product (3 x 3 x 3 = 27 tests)
number of tests: 27
--- run ---
(temp=300, pressure=100, coolant_loss=10) -> OK
(temp=300, pressure=100, coolant_loss=35) -> WARN
(temp=300, pressure=100, coolant_loss=75) -> CRITICAL
(temp=300, pressure=210, coolant_loss=10) -> WARN
(temp=300, pressure=210, coolant_loss=35) -> WARN
(temp=300, pressure=210, coolant_loss=75) -> CRITICAL
(temp=300, pressure=270, coolant_loss=10) -> CRITICAL
(temp=300, pressure=270, coolant_loss=35) -> CRITICAL
(temp=300, pressure=270, coolant_loss=75) -> CRITICAL
(temp=700, pressure=100, coolant_loss=10) -> WARN
(temp=700, pressure=100, coolant_loss=35) -> WARN
(temp=700, pressure=100, coolant_loss=75) -> CRITICAL
(temp=700, pressure=210, coolant_loss=10) -> WARN
(temp=700, pressure=210, coolant_loss=35) -> WARN
(temp=700, pressure=210, coolant_loss=75) -> CRITICAL
(temp=700, pressure=270, coolant_loss=10) -> CRITICAL
(temp=700, pressure=270, coolant_loss=35) -> CRITICAL
(temp=700, pressure=270, coolant

### Step 2 — reduce to a pairwise (2-way) set — exercise

A **pairwise** (a.k.a. **2-way**) test set covers, for **every pair of factors**, **all** combinations of their levels — without necessarily covering every 3-way combination.

For 3 factors × 3 levels:
- full grid = **27** tests,
- 2-way pairs to cover = 3 factor-pairs × (3 × 3) = **27 pairs**,
- a perfect pairwise set needs only **9** tests.

**TODO** — In the next cell, fill `pairwise_set` with combinations (each value taken from the EP levels above) until **every** 2-way combination is covered. The starter set below is **deliberately incomplete**. Run the **verification** cell (Step 3), read which pairs you still miss, add rows, and repeat. Try to reach **9** tests.

In [17]:
# TODO: choose a SMALL subset of (temp, pressure, coolant_loss) combinations
# that still covers EVERY 2-way combination of factor levels.
#
# Each value MUST come from the EP levels defined above:
#   temp         in [300, 700, 900]
#   pressure     in [100, 210, 270]
#   coolant_loss in [10,  35,  75]
#
# Full grid = 27 tests. A perfect pairwise set needs only 9.
# This starter is INCOMPLETE on purpose — run the verification cell below,
# read the missing pairs, add rows, and iterate until coverage is 100%.

pairwise_set: list[tuple[int, int, int]] = [
    # (temp, pressure, coolant_loss)
    (300, 100, 10),
    (300, 210, 35),
    (300, 270, 75),
    (700, 100, 35),
    # ... add more rows until every 2-way pair is covered (aim for 9) ...
]

run_grid(pairwise_set, "CIT — candidate pairwise set")

# ----------------------------------------------------------------------
# Reference solution (an L9 orthogonal array). Try it yourself first!
# Uncomment to compare — it covers all 27 pairs with exactly 9 tests.
# pairwise_set = [
#     (300, 100, 10), (300, 210, 35), (300, 270, 75),
#     (700, 100, 35), (700, 210, 75), (700, 270, 10),
#     (900, 100, 75), (900, 210, 10), (900, 270, 35),
# ]


CIT — candidate pairwise set
number of tests: 4
--- run ---
(temp=300, pressure=100, coolant_loss=10) -> OK
(temp=300, pressure=210, coolant_loss=35) -> WARN
(temp=300, pressure=270, coolant_loss=75) -> CRITICAL
(temp=700, pressure=100, coolant_loss=35) -> WARN



### Step 3 — verify 2-way coverage

We verify in two simple steps:

1. **Build the 2-way coverage set** — for **every pair of factors**, list **all** combinations of their levels. With 3 factors of 3 levels each that is `temp×pressure` + `temp×coolant_loss` + `pressure×coolant_loss` = 3 × (3 × 3) = **27** required pairs.

   *Illustration:* if the levels were `a = [1,2,3]`, `b = [4,5,6]`, `c = [7,8,9]`, the coverage set is **every** `(a,b)` pair, **every** `(a,c)` pair, and **every** `(b,c)` pair — `(1,4),(1,5),…,(3,6)`, `(1,7),…,(3,9)`, `(4,7),…,(6,9)`.

2. **Check each required pair** with a simple loop and print it **one by one** with **`O`** (present in at least one `(temp, pressure, coolant_loss)` triple of `pairwise_set`) or **`X`** (still missing). All `O` → **PASS**; any `X` → **FAIL**.

Keep editing `pairwise_set` in Step 2 until every line is **`O`** — ideally with only **9** tests.

In [18]:
# --- Step 3a: build the 2-way coverage set --------------------------------
# For every pair of factors, list ALL combinations of their levels.
# Each required item is (i, j, value_i, value_j), where i, j are positions
# in the (temp, pressure, coolant_loss) triple.
names  = ["temp", "pressure", "coolant_loss"]
levels = [factors["temp"], factors["pressure"], factors["coolant_loss"]]

coverage_set: list[tuple[int, int, int, int]] = []
for i, j in [(0, 1), (0, 2), (1, 2)]:          # (a,b), (a,c), (b,c)
    for vi in levels[i]:
        for vj in levels[j]:
            coverage_set.append((i, j, vi, vj))

# --- Step 3b: check each required pair and mark it O (covered) / X (missing)
print("2-way coverage check   (O = covered, X = missing)")

covered = 0
prev_pair = None
for i, j, vi, vj in coverage_set:
    if (i, j) != prev_pair:                     # new factor-pair -> print a header
        print(f"\n[ {names[i]} x {names[j]} ]")
        prev_pair = (i, j)
    # is this pair present in at least one test triple of the suite?
    hit = False
    for t in pairwise_set:
        if t[i] == vi and t[j] == vj:
            hit = True
            break
    if hit:
        covered += 1
    print(f"  {'O' if hit else 'X'}  ({names[i]}={vi}, {names[j]}={vj})")

# --- result ---------------------------------------------------------------
print("\n" + "-" * 40)
print(f"covered: {covered}/{len(coverage_set)}   "
      f"(tests in suite = {len(pairwise_set)}, full grid = 27)")
if covered == len(coverage_set):
    print("RESULT: PASS  - every 2-way combination is covered. ✅")
else:
    print("RESULT: FAIL  - fill the X pairs by adding combinations in Step 2.")


2-way coverage check   (O = covered, X = missing)

[ temp x pressure ]
  O  (temp=300, pressure=100)
  O  (temp=300, pressure=210)
  O  (temp=300, pressure=270)
  O  (temp=700, pressure=100)
  X  (temp=700, pressure=210)
  X  (temp=700, pressure=270)
  X  (temp=900, pressure=100)
  X  (temp=900, pressure=210)
  X  (temp=900, pressure=270)

[ temp x coolant_loss ]
  O  (temp=300, coolant_loss=10)
  O  (temp=300, coolant_loss=35)
  O  (temp=300, coolant_loss=75)
  X  (temp=700, coolant_loss=10)
  O  (temp=700, coolant_loss=35)
  X  (temp=700, coolant_loss=75)
  X  (temp=900, coolant_loss=10)
  X  (temp=900, coolant_loss=35)
  X  (temp=900, coolant_loss=75)

[ pressure x coolant_loss ]
  O  (pressure=100, coolant_loss=10)
  O  (pressure=100, coolant_loss=35)
  X  (pressure=100, coolant_loss=75)
  X  (pressure=210, coolant_loss=10)
  O  (pressure=210, coolant_loss=35)
  X  (pressure=210, coolant_loss=75)
  X  (pressure=270, coolant_loss=10)
  X  (pressure=270, coolant_loss=35)
  O  (pressu